In [ ]:
######################## Multiple PDF processing start ####################

In [1]:
import os
import glob
import re
import pickle
# import faiss
import chromadb
from dotenv import load_dotenv

from llama_parse import LlamaParse
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.docstore.in_memory import InMemoryDocstore
# from langchain.agents import initialize_agent, Tool, AgentType
# from langchain.memory import ConversationBufferMemory
from langchain_openai import AzureOpenAIEmbeddings, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

/var/folders/by/lbg2h1fn4gj5dzkqpzh3133h0000gp/T/ipykernel_18470/2053777685.py:9: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse
/opt/miniconda3/envs/llm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import nest_asyncio

nest_asyncio.apply()

In [2]:
load_dotenv()
LLAMA_API_KEY = os.getenv("LLAMA_CLOUD_API_KEY")
os.environ["AZURE_OPENAI_API_KEY"] = os.getenv("AZURE_OPENAI_API_KEY")
os.environ["AZURE_OPENAI_ENDPOINT"] = os.getenv("AZURE_OPENAI_ENDPOINT")
os.environ["openai_api_key"] = os.getenv("openai_api_key")

In [ ]:
folder_path = './app/data/input_data/pdf/'
output_folder_path = './app/data/input_data/markdown_test/'

parser = LlamaParse(
    api_key= LLAMA_API_KEY,
    result_type = 'markdown'
)

for file_name in os.listdir(folder_path):
    if file_name.endswith('.pdf'):
        full_path = os.path.join(folder_path, file_name)
        document = parser.load_data(full_path)

        os.makedirs(output_folder_path, exist_ok=True)
        output_file_path = os.path.join(output_folder_path, file_name.replace('.pdf', '-restructured.md'))

        with open(output_file_path, 'w') as f:
            for doc in document:
                f.write(doc.text + '\n\n')


Started parsing the file under job_id a755e7f2-bd9a-42a6-b94b-f61ea3169b2a


In [19]:
md_folder_path = './app/data/input_data/markdown_test/'
all_documents = []

for file_name in os.listdir(md_folder_path):
    if file_name.endswith('.md'):
        full_path = os.path.join(md_folder_path, file_name)
        docs = UnstructuredMarkdownLoader(full_path).load()

        for doc in docs:
        # Extract company and plan information from the filename.
            # Assumes filename pattern: policy-<company>-plan<plan_number>-restructured.md
            match = re.search(r'policy-([a-zA-Z_]+)-plan-([a-zA-Z_]+)', file_name, re.IGNORECASE)
            if match:
                doc.metadata["company"] = match.group(1).upper()
                doc.metadata["plan"] = match.group(2)
            else:
                doc.metadata["source_file"] = file_name
        all_documents.extend(docs)
        

In [20]:
# --------------------------------------------------
# 3. Split the loaded documents into chunks
# --------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs_split = text_splitter.split_documents(all_documents)

In [ ]:
# # check the chunks have correct metadata
# docs_split[600].metadata

In [24]:
# --------------------------------------------------
# 4. Create embeddings using embeddings from Azure OpenAI
# --------------------------------------------------
embedding_model = AzureOpenAIEmbeddings(
    azure_deployment="text-embedding-3-small",  # Your Azure deployment name for embeddings
    openai_api_version="2024-02-01",
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"]
)

In [ ]:
# Define the OpenAI embedding model
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",  # You can choose a different model if needed
    openai_api_key=os.environ["openai_api_key"],
)

In [26]:
vector_store = Chroma.from_documents(
    documents=docs_split,  # List of document chunks
    embedding=embedding_model, 
    persist_directory="./app/data/vector_data/chroma_db",  # Path to persist data
    collection_name="insurance_policies"  # Name of the collection
)

# Persist the data
vector_store.persist()


In [ ]:
######################## Multiple PDF processing End ####################

In [ ]:
# # Create the vector store
# vector_store = FAISS.from_documents(docs_split, embedding_model)

# # Save FAISS index and document store
# faiss.write_index(vector_store.index, "./data/vector_data/faiss_index")
# with open("./data/vector_data/faiss_documents.pkl", "wb") as f:
#     pickle.dump(vector_store.docstore._dict, f)